<img src="MUML_header.svg" alt="Header" style="width:100%; height:auto;">

### Character-level Language Modeling with MLPs

You have to implement a simple character-level language model using multi-layer perceptrons (MLPs) in PyTorch. The goal is to predict the next character in a text sequence given a fixed-length context of previous characters. You will implement and compare two different input encodings:

*   One-hot encoding of characters
*   Learned embeddings for characters

In [1]:
import os
import math
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

You are going to work with the text "PrideAndPrejudicePreface.txt". Convert all characters to lowercase for simplicity.

You have to build a vocabulary of unique characters in the text, and assign an integer ID to each one. Based on this assignment, create two functions:
* encode receive a text and returns a list of integers
* decode receive a list of integers and return a string

In [2]:
file_name = "PrideAndPrejudicePreface.txt"
file = open(file_name,"r", encoding="utf-8").read().lower()

In [3]:
def encode(text):
    unique_characters = sorted(set(file))
    vocabulary = {char: i for i, char in enumerate(unique_characters)}
    list_integers = [vocabulary[char] for char in text]
    return list_integers

def decode(list_integers):
    unique_characters = sorted(set(file))  
    inverse_vocabulary = {i: char for i, char in enumerate(unique_characters)}
    text = "".join([inverse_vocabulary[i] for i in list_integers])
    return text

In [4]:
encode('malena es un cuy')
decode([28, 16, 27, 20, 29, 16, 1, 20, 34, 1, 36, 29, 1, 18, 36, 40])

'malena es un cuy'

Define two custom PyTorch Dataset classes, CharDatasetOneHot and CharDatasetEmbed, that take as input the text and a fixed context size (e.g., 5 characters). For each position, they return a context (the previous context_size characters) and a target (the next character).

For the one-hot version, the function has to convert the context indices into one-hot vectors and flatten them into a single vector of size context_size x vocab_size.

For the embedding version the function has to return the context indices directly (as integers).

In [5]:

class CharDatasetOneHot(Dataset):
    def __init__(self, input_text, context_size):
        self.context_size = context_size
        self.encoded_text = encode(input_text)
        self.vocab_size = len(set(file))

    def __len__(self):
        return len(self.encoded_text) - self.context_size

    def __getitem__(self, idx):
        context_indices = self.encoded_text[idx : idx + self.context_size]
        target_index = self.encoded_text[idx + self.context_size]
        
        context_tensor = torch.tensor(context_indices, dtype=torch.long)
        target_tensor = torch.tensor(target_index, dtype=torch.long)

        context_one_hot = F.one_hot(context_tensor, num_classes=self.vocab_size).float()
        context_flattened = context_one_hot.view(-1)
        
        return context_flattened, target_tensor

In [6]:
class CharDatasetEmbed(Dataset):
    def __init__(self, input_text, context_size):
        self.context_size = context_size
        self.encoded_text = encode(input_text)

    def __len__(self):
        return len(self.encoded_text) - self.context_size

    def __getitem__(self, idx):
        context_indices = self.encoded_text[idx : idx + self.context_size]
        target_index = self.encoded_text[idx + self.context_size]

        context_tensor = torch.tensor(context_indices, dtype=torch.long)
        target_tensor = torch.tensor(target_index, dtype=torch.long)

        return context_tensor, target_tensor

You will implement two different models:

1. One-hot MLP model
    * Input size = context_size * vocab_size
    * Hidden layers: you will experiment with different number and sizes
    * Output layer: Linear → size = vocab_size (one logit per character).
2. Embedding MLP model
    * Use nn.Embedding(vocab_size, embed_dim) to map character indices to dense vectors.
    * Flatten embeddings for the context into a single vector of size (context_size * embed_dim).
    * Pass this vector through an MLP similar to the one-hot version.

In [7]:
class OneHotMLP(nn.Module):
    def __init__(self, vocab_size, context_size, hidden_dims):
        super().__init__()
        self.input_size = context_size * vocab_size

        layers = []
        prev_dim = self.input_size
        
        for hidden in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden))
            layers.append(nn.ReLU())
            prev_dim = hidden
        
        self.hidden_layers = nn.Sequential(*layers)
        self.output_layer = nn.Linear(prev_dim, vocab_size)

    def forward(self, x):
        x = self.hidden_layers(x)
        logits = self.output_layer(x)
        return logits
        
        

In [8]:
class EmbeddingMLP(nn.Module):
    def __init__(self, vocab_size, context_size, hidden_dims, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.input_size = context_size * embed_dim

        layers = []
        prev_dim = self.input_size
        
        for hidden in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden))
            layers.append(nn.ReLU())
            prev_dim = hidden
        
        self.hidden_layers = nn.Sequential(*layers)
        self.output_layer = nn.Linear(prev_dim, vocab_size)

    def forward(self, x):
        embeds = self.embedding(x)  
        flatten_embedding = embeds.view(x.size(0), -1)  # shape: [batch_size, context_size * embed_dim]
        hidden_out = self.hidden_layers(flatten_embedding)
        logits = self.output_layer(hidden_out)
        return logits

For both training, use CrossEntropyLoss as the loss function and Adam optimizer. Train with minibatches using DataLoader.

Print the average loss at the end of each epoch.

In [9]:
context_size = 5
batch_size = 32
epochs = 10
learning_rate = 0.001
hidden_dims = [128, 64]
embed_dim = 16

vocab_size = len(set(file))

dataset_onehot = CharDatasetOneHot(file, context_size)
dataset_embed = CharDatasetEmbed(file, context_size)

loader_onehot = DataLoader(dataset_onehot, batch_size=batch_size, shuffle=True)
loader_embed = DataLoader(dataset_embed, batch_size=batch_size, shuffle=True)

In [10]:
def train_model(model, dataloader, epochs, lr, model_name):
    print(f"\n--- Starting training: {model_name} ---")
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for x_batch, y_batch in dataloader:
            optimizer.zero_grad()
            
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch [{epoch}/{epochs}] - Avg Loss: {avg_loss:.4f}")

In [11]:
# Instanciar y entrenar modelo OneHotMLP
model_onehot = OneHotMLP(vocab_size, context_size, hidden_dims)
train_model(model_onehot, loader_onehot, epochs, learning_rate, "OneHotMLP")

# Instanciar y entrenar modelo EmbeddingMLP
model_embed = EmbeddingMLP(vocab_size, context_size, hidden_dims, embed_dim)
train_model(model_embed, loader_embed, epochs, learning_rate, "EmbeddingMLP")


--- Starting training: OneHotMLP ---
Epoch [1/10] - Avg Loss: 2.6257
Epoch [2/10] - Avg Loss: 2.2111
Epoch [3/10] - Avg Loss: 2.0723
Epoch [4/10] - Avg Loss: 1.9778
Epoch [5/10] - Avg Loss: 1.8976
Epoch [6/10] - Avg Loss: 1.8292
Epoch [7/10] - Avg Loss: 1.7697
Epoch [8/10] - Avg Loss: 1.7177
Epoch [9/10] - Avg Loss: 1.6714
Epoch [10/10] - Avg Loss: 1.6323

--- Starting training: EmbeddingMLP ---
Epoch [1/10] - Avg Loss: 2.5639
Epoch [2/10] - Avg Loss: 2.1643
Epoch [3/10] - Avg Loss: 2.0155
Epoch [4/10] - Avg Loss: 1.9147
Epoch [5/10] - Avg Loss: 1.8358
Epoch [6/10] - Avg Loss: 1.7718
Epoch [7/10] - Avg Loss: 1.7232
Epoch [8/10] - Avg Loss: 1.6758
Epoch [9/10] - Avg Loss: 1.6355
Epoch [10/10] - Avg Loss: 1.6020


Implement a function compute_perplexity(model, dataset) that returns the perplexity of the model on a given dataset.

In [12]:
def compute_perplexity(model, dataset):
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    criterion = nn.CrossEntropyLoss()

    model.eval()
    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            logits = model(x_batch)
            loss = criterion(logits, y_batch)  

            total_loss += loss.item() * y_batch.size(0)
            total_samples += y_batch.size(0)

    avg_loss = total_loss / total_samples

    return math.exp(avg_loss)

Implement a function generate(model, context, length) that generates new text given an initial context.

Test your model by generating text after training.

You must run and compare different experiments:

* Encoding methods: one-hot vs. embeddings
* Different context sizes
* Several MLP architectures (number of hidden layers and their sizes)

For each experiment, analyze
* the training loss per epoch
* the final perplexity
* the generated text

In [13]:
import torch
import torch.nn.functional as F

def generate(model, context, length):
    model.eval()
    
    unique_characters = sorted(set(file))
    vocab_size = len(unique_characters)
    encoded_text = encode(context)
    
    if isinstance(model, OneHotMLP):
        context_size = model.input_size // vocab_size 
    else: # EmbeddingMLP
        context_size = model.input_size // model.embedding.embedding_dim

    if len(encoded_text) < context_size:
        raise ValueError(f"The initial contex should have at least {context_size} characters.")

    with torch.no_grad():
        for _ in range(length):
            current_context = encoded_text[-context_size:]
            
            if isinstance(model, OneHotMLP):
                context_tensor = torch.tensor(current_context, dtype=torch.long)
                context_one_hot = F.one_hot(context_tensor, num_classes=vocab_size).float().view(1, -1)
                logits = model(context_one_hot)
                               
            else: # EmbeddingMLP
                context_tensor = torch.tensor(current_context, dtype=torch.long).unsqueeze(0)
                logits = model(context_tensor)
            
            predicted_index = torch.argmax(logits, dim=-1).item()
            encoded_text.append(predicted_index)

    return decode(encoded_text)

In [18]:
experiments = [
    # 1. Comparación One-Hot vs. Embeddings (Mismo contexto y arquitectura)
    {"name": "Exp 1: One-Hot (ctx=5, [128, 64])", "type": "one_hot", "context_size": 5, "hidden_dims": [128, 64], "embed_dim": None},
    {"name": "Exp 2: Embedding (ctx=5, [128, 64], emb=16)", "type": "embed", "context_size": 5, "hidden_dims": [128, 64], "embed_dim": 16},
    
    # 2. Variación de Tamaño de Contexto (Usando Embeddings)
    {"name": "Exp 3: Embedding (ctx=3, [128, 64], emb=16)", "type": "embed", "context_size": 3, "hidden_dims": [128, 64], "embed_dim": 16},
    {"name": "Exp 4: Embedding (ctx=10, [128, 64], emb=16)", "type": "embed", "context_size": 10, "hidden_dims": [128, 64], "embed_dim": 16},
    
    # 3. Variación de Arquitectura MLP (Profundidad y Ancho)
    {"name": "Exp 5: Embedding Pequeño (ctx=5, [64], emb=16)", "type": "embed", "context_size": 5, "hidden_dims": [64], "embed_dim": 16},
    {"name": "Exp 6: Embedding Profundo (ctx=5, [256, 128, 64], emb=16)", "type": "embed", "context_size": 5, "hidden_dims": [256, 128, 64], "embed_dim": 16},
]

# Parámetros de entrenamiento
epochs = 10
batch_size = 64
learning_rate = 0.001
prompt_text = "hello to everyone "  # Asegurar que sea >= max context_size usado en los experimentos
vocab_size = len(set(file))

results = []

In [19]:
# --------------------------------------------------------------------------
# Bucle de Ejecución
# --------------------------------------------------------------------------
for exp in experiments:
    print(f"\n==========================================")
    print(f"Ejecutando: {exp['name']}")
    print(f"==========================================")
    
    # Instanciación de Dataset y Modelo
    if exp["type"] == "one_hot":
        ds = CharDatasetOneHot(file, exp["context_size"])
        model = OneHotMLP(vocab_size, exp["context_size"], exp["hidden_dims"])
    else:
        ds = CharDatasetEmbed(file, exp["context_size"])
        model = EmbeddingMLP(vocab_size, exp["context_size"], exp["hidden_dims"], exp["embed_dim"])
    
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    # Entrenamiento y registro de pérdidas
    history_loss = []
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for x_b, y_b in loader:
            optimizer.zero_grad()
            logits = model(x_b)
            loss = criterion(logits, y_b)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        avg_loss = total_loss / len(loader)
        history_loss.append(avg_loss)
        print(f"Epoch {epoch:02d}/{epochs:02d} | Training Loss: {avg_loss:.4f}")
    
    # Cálculo de Perplejidad Final
    perplexity = compute_perplexity(model, ds)
    
    # Generación de Texto
    # Ajustamos el prompt para garantizar que cumpla la longitud mínima necesaria
    test_prompt = prompt_text if len(prompt_text) >= exp["context_size"] else prompt_text.zfill(exp["context_size"])
    generated_output = generate(model, test_prompt, length=50)
    
    # Guardar resultados
    results.append({
        "name": exp["name"],
        "final_loss": history_loss[-1],
        "perplexity": perplexity,
        "sample": generated_output
    })

# --------------------------------------------------------------------------
# Resumen Comparativo de Resultados
# --------------------------------------------------------------------------
print("\n" + "="*80)
print(f"{'Experimento':<45} | {'Loss Final':<10} | {'Perplejidad':<12}")
print("="*80)
for res in results:
    print(f"{res['name']:<45} | {res['final_loss']:<10.4f} | {res['perplexity']:<12.4f}")
print("="*80)


Ejecutando: Exp 1: One-Hot (ctx=5, [128, 64])
Epoch 01/10 | Training Loss: 2.7502
Epoch 02/10 | Training Loss: 2.2936
Epoch 03/10 | Training Loss: 2.1675
Epoch 04/10 | Training Loss: 2.0800
Epoch 05/10 | Training Loss: 2.0103
Epoch 06/10 | Training Loss: 1.9487
Epoch 07/10 | Training Loss: 1.8931
Epoch 08/10 | Training Loss: 1.8447
Epoch 09/10 | Training Loss: 1.8018
Epoch 10/10 | Training Loss: 1.7628

Ejecutando: Exp 2: Embedding (ctx=5, [128, 64], emb=16)
Epoch 01/10 | Training Loss: 2.6594
Epoch 02/10 | Training Loss: 2.2439
Epoch 03/10 | Training Loss: 2.0949
Epoch 04/10 | Training Loss: 1.9934
Epoch 05/10 | Training Loss: 1.9167
Epoch 06/10 | Training Loss: 1.8501
Epoch 07/10 | Training Loss: 1.7993
Epoch 08/10 | Training Loss: 1.7556
Epoch 09/10 | Training Loss: 1.7144
Epoch 10/10 | Training Loss: 1.6814

Ejecutando: Exp 3: Embedding (ctx=3, [128, 64], emb=16)
Epoch 01/10 | Training Loss: 2.6142
Epoch 02/10 | Training Loss: 2.2125
Epoch 03/10 | Training Loss: 2.0964
Epoch 04/10

In [20]:
# --------------------------------------------------------------------------
# Resumen Comparativo de Resultados
# --------------------------------------------------------------------------
print("\n" + "=" * 90)
print(
    f"{'Experimento':<45} | {'Loss Final':<10} | {'Perplejidad':<12}"
)
print("=" * 90)

for res in results:
    print(
        f"{res['name']:<45} | {res['final_loss']:<10.4f} | {res['perplexity']:<12.4f}"
    )
    # Limpiamos saltos de línea del texto generado para imprimirlo de forma limpia
    clean_sample = res["sample"].replace("\n", " ")
    print(f"  └─ Texto generado: \"{clean_sample}\"")
    print("-" * 90)


Experimento                                   | Loss Final | Perplejidad 
Exp 1: One-Hot (ctx=5, [128, 64])             | 1.7628     | 5.4983      
  └─ Texto generado: "hello to everyone of the mare of the mare of the mare of the mare of"
------------------------------------------------------------------------------------------
Exp 2: Embedding (ctx=5, [128, 64], emb=16)   | 1.6814     | 5.0553      
  └─ Texto generado: "hello to everyone of the common and that the common and that the com"
------------------------------------------------------------------------------------------
Exp 3: Embedding (ctx=3, [128, 64], emb=16)   | 1.8046     | 5.8305      
  └─ Texto generado: "hello to everyone is austen a compers austen a compers austen a comp"
------------------------------------------------------------------------------------------
Exp 4: Embedding (ctx=10, [128, 64], emb=16)  | 1.5824     | 4.3530      
  └─ Texto generado: "hello to everyone of miss austen ther shode is not a she o